# 命令セットへのコンパイル

この章では、`qret compile` で回路を `SC_LS_FIXED_V0` へ変換する手順を確認します。
生成した pipeline state JSON は、次章の `profile` と可視化で使います。

本章で確認する内容は次の 4 点です。

1. `compile` の基本実行手順
2. `Dim2` / `Dim3` / `DistributedDim2` の違い
3. `PBC` の有効化方法
4. PBC で必要な入力条件（末尾で全量子ビット測定）


In [1]:
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../..").resolve()
if platform.system() == "Windows":
    qret_path = project_root / "build" / "bin"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth.exe"
elif platform.system() == "Darwin": 
    qret_path = project_root / "build" / "main"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
elif platform.system() == "Linux":
    qret_path = project_root / "build" / "main"
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"
else:
    raise ValueError("Unsupported platform")
os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")


## 0. 事前確認
最初に CLI が起動できることを確認します。

In [2]:
!qret --version

Quration: Quantum Resource Estimation Toolchain for FTQC version 1.0.0


## 入力回路（通常モード）
この章の Dim2 / Dim3 / DistributedDim2 では `data/tutorial_5.qasm` を使います。

In [3]:
Code(filename="data/tutorial_5.qasm", language="ASM")

OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];
h q[0];
h q[1];
h q[2];
h q[3];
t q[0];
cx q[0],q[1];
t q[1];

## qret compile
`compile` は source (`IR`/`OpenQASM2`/`SC_LS_FIXED_V0`) を読み取り、
対象 ISA (`SC_LS_FIXED_V0`) の命令列へ変換します。

In [4]:
!qret compile --help


qret 'compile' options:
  -h [ --help ]                         Show this help and exit.
  --help-hidden                         Show help including hidden options.
  --quiet                               Suppress non-error output.
  --verbose                             Enable verbose logging (more detail than default).
  --debug                               Enable debug logging (most detailed; implies --verbose).
  --color                               Enable colored output.
  --pipeline FILE                       Path to a pipeline specification file.
  -i [ --input ] FILE                   Path to the input file.
  -f [ --function ] NAME                [source=IR] Name of the function to compile.
  -o [ --output ] FILE (=a.json)        Path to the output SC_LS_FIXED_V0 file.
  -s [ --source ] KIND (=IR)            Source representation: 'IR', 'OpenQASM2', or 'SC_LS_FIXED_V0'.
  -t [ --target ] KIND (=SC_LS_FIXED_V0)
                                        Target machine name.
  -

## パイプラインファイルを読む
各 YAML には `source` / `input` / `output` / `sc_ls_fixed_v0_topology` / `sc_ls_fixed_v0_machine_type` / `pass` などの実行条件を定義します。
各モードでファイルを読み分けながら、設定値の差分を確認します。


### Machine Type と PBC モード
- `Dim2`: 単一平面
- `Dim3`: 3D 配置
- `DistributedDim2`: 複数チップ(平面)
- `DistributedDim3`: 現在未対応
- `PBC`: `sc_ls_fixed_v0_machine_type` ではなく `sc_ls_fixed_v0_enable_pbc_mode` で有効化
  - 現在の実装では `Dim2` のみ対応
  - 入力回路は末尾で全量子ビット測定が必要


## Dim2 へのコンパイル

In [5]:
dim2_topology_path = "data/tutorial_5_dim2_topology.yaml"
dim2_pipeline_path = "data/tutorial_5_dim2_pipeline.yaml"

### チップファイル

In [6]:
Code(filename=dim2_topology_path, language="YAML")

grids:
  - type: plane
    coord: [10, 10, 0]
    magic_factory:
      - symbol: 0
        coord: [0, 0]
      - symbol: 1
        coord: [0, 1]
      - symbol: 2
        coord: [0, 2]
      - symbol: 3
        coord: [0, 3]

### パイプライン

In [7]:
Code(filename=dim2_pipeline_path, language="YAML")

source: OpenQASM2
input: data/tutorial_5.qasm
function: main
target: SC_LS_FIXED_V0
output: tutorial_5_dim2.json
sc_ls_fixed_v0_topology: data/tutorial_5_dim2_topology.yaml
sc_ls_fixed_v0_machine_type: Dim2
sc_ls_fixed_v0_magic_generation_period: 15
sc_ls_fixed_v0_maximum_magic_state_stock: 10000
sc_ls_fixed_v0_entanglement_generation_period: 100
sc_ls_fixed_v0_maximum_entangled_state_stock: 10
sc_ls_fixed_v0_reaction_time: 1
sc_ls_fixed_v0_drop_rate: 0.1
sc_ls_fixed_v0_code_cycle_time_sec: 0.00000001
sc_ls_fixed_v0_physical_error_rate: 0.01
sc_ls_fixed_v0_allowed_failure_prob: 0.01
sc_ls_fixed_v0_pass:
  - sc_ls_fixed_v0::init_compile_info
  - sc_ls_fixed_v0::mapping
  - sc_ls_fixed_v0::routing

### 実行

In [8]:
!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret asm -i tutorial_5_dim2.json -o tutorial_5_dim2.asm --print-metadata 1

2026-03-03 11:09:05 - INFO  - Load OpenQASM2.
2026-03-03 11:09:05 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:05 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:05 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:05 - INFO  - Run passes.
2026-03-03 11:09:05 - INFO  - Run InitCompileInfo
2026-03-03 11:09:05 - INFO  - Initialize compile information
2026-03-03 11:09:05 - INFO  - Run Mapping
2026-03-03 11:09:05 - INFO  - Run Routing
2026-03-03 11:09:05 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-03 11:09:05 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_dim2.json


## Dim3 へのコンパイル

In [9]:
dim3_topology_path = "data/tutorial_5_dim3_topology.yaml"
dim3_pipeline_path = "data/tutorial_5_dim3_pipeline.yaml"

### チップファイル

In [10]:
Code(filename=dim3_topology_path, language="YAML")

grids:
  - type: grid
    coord: [5, 5, 0, 4]
    magic_factory:
      - symbol: 0
        coord: [0, 0, 0]
      - symbol: 1
        coord: [0, 0, 1]
      - symbol: 2
        coord: [0, 0, 2]
      - symbol: 3
        coord: [0, 0, 3]

### パイプライン

In [11]:
Code(filename=dim3_pipeline_path, language="YAML")

source: OpenQASM2
input: data/tutorial_5.qasm
function: main
target: SC_LS_FIXED_V0
output: tutorial_5_dim3.json
sc_ls_fixed_v0_topology: data/tutorial_5_dim3_topology.yaml
sc_ls_fixed_v0_machine_type: Dim3
sc_ls_fixed_v0_magic_generation_period: 15
sc_ls_fixed_v0_maximum_magic_state_stock: 10000
sc_ls_fixed_v0_entanglement_generation_period: 100
sc_ls_fixed_v0_maximum_entangled_state_stock: 10
sc_ls_fixed_v0_reaction_time: 1
sc_ls_fixed_v0_drop_rate: 0.1
sc_ls_fixed_v0_code_cycle_time_sec: 0.00000001
sc_ls_fixed_v0_physical_error_rate: 0.01
sc_ls_fixed_v0_allowed_failure_prob: 0.01
sc_ls_fixed_v0_pass:
  - sc_ls_fixed_v0::init_compile_info
  - sc_ls_fixed_v0::mapping
  - sc_ls_fixed_v0::routing

### 実行

In [12]:
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret asm -i tutorial_5_dim3.json -o tutorial_5_dim3.asm --print-metadata 1

2026-03-03 11:09:06 - INFO  - Load OpenQASM2.
2026-03-03 11:09:06 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:06 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:06 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:06 - INFO  - Run passes.
2026-03-03 11:09:06 - INFO  - Run InitCompileInfo
2026-03-03 11:09:06 - INFO  - Initialize compile information
2026-03-03 11:09:06 - INFO  - Run Mapping
2026-03-03 11:09:06 - INFO  - Run Routing
2026-03-03 11:09:06 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-03 11:09:06 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_dim3.json


## マルチノード（DistributedDim2）へのコンパイル

In [13]:
dist_topology_path = "data/tutorial_5_dist_topology.yaml"
dist_pipeline_path = "data/tutorial_5_dist_pipeline.yaml"

### チップファイル

In [14]:
Code(filename=dist_topology_path, language="YAML")

grids:
  - type: plane
    coord: [10, 10, 0]
    magic_factory:
      - symbol: 0
        coord: [0, 4]
    entanglement_factory:
      - symbol: 0
        pair: 1
        coord: [0, 0]
      - symbol: 7
        pair: 6
        coord: [0, 1]
    qubit:
      - symbol: 0
        coord: [2, 2]
      - symbol: 4
        coord: [2, 4]
      - symbol: 8
        coord: [2, 6]
      - symbol: 12
        coord: [2, 8]
      - symbol: 16
        coord: [4, 2]
      - symbol: 20
        coord: [4, 4]
      - symbol: 24
        coord: [4, 6]
      - symbol: 28
        coord: [4, 8]
      - symbol: 32
        coord: [6, 2]
      - symbol: 36
        coord: [6, 4]
      - symbol: 40
        coord: [6, 6]
      - symbol: 44
        coord: [6, 8]
      - symbol: 48
        coord: [8, 2]
      - symbol: 52
        coord: [8, 4]
      - symbol: 56
        coord: [8, 6]
      - symbol: 60
        coord: [8, 8]
  - type: plane
    coord: [10, 10, 2]
    magic_factory:
      - symbol: 1
        coord: [0, 4]
    entanglement_factory:
      - symbol: 1
        pair: 0
        coord: [0, 0]
      - symbol: 2
        pair: 3
        coord: [0, 1]
    qubit:
      - symbol: 1
        coord: [2, 2]
      - symbol: 5
        coord: [2, 4]
      - symbol: 9
        coord: [2, 6]
      - symbol: 13
        coord: [2, 8]
      - symbol: 17
        coord: [4, 2]
      - symbol: 21
        coord: [4, 4]
      - symbol: 25
        coord: [4, 6]
      - symbol: 29
        coord: [4, 8]
      - symbol: 33
        coord: [6, 2]
      - symbol: 37
        coord: [6, 4]
      - symbol: 41
        coord: [6, 6]
      - symbol: 45
        coord: [6, 8]
      - symbol: 49
        coord: [8, 2]
      - symbol: 53
        coord: [8, 4]
      - symbol: 57
        coord: [8, 6]
      - symbol: 61
        coord: [8, 8]
  - type: plane
    coord: [10, 10, 4]
    magic_factory:
      - symbol: 2
        coord: [0, 4]
    entanglement_factory:
      - symbol: 3
        pair: 2
        coord: [0, 0]
      - symbol: 4
        pair: 5
        coord: [0, 1]
    qubit:
      - symbol: 2
        coord: [2, 2]
      - symbol: 6
        coord: [2, 4]
      - symbol: 10
        coord: [2, 6]
      - symbol: 14
        coord: [2, 8]
      - symbol: 18
        coord: [4, 2]
      - symbol: 22
        coord: [4, 4]
      - symbol: 26
        coord: [4, 6]
      - symbol: 30
        coord: [4, 8]
      - symbol: 34
        coord: [6, 2]
      - symbol: 38
        coord: [6, 4]
      - symbol: 42
        coord: [6, 6]
      - symbol: 46
        coord: [6, 8]
      - symbol: 50
        coord: [8, 2]
      - symbol: 54
        coord: [8, 4]
      - symbol: 58
        coord: [8, 6]
      - symbol: 62
        coord: [8, 8]
  - type: plane
    coord: [10, 10, 6]
    magic_factory:
      - symbol: 3
        coord: [0, 4]
    entanglement_factory:
      - symbol: 5
        pair: 4
        coord: [0, 0]
      - symbol: 6
        pair: 7
        coord: [0, 1]
    qubit:
      - symbol: 3
        coord: [2, 2]
      - symbol: 7
        coord: [2, 4]
      - symbol: 11
        coord: [2, 6]
      - symbol: 15
        coord: [2, 8]
      - symbol: 19
        coord: [4, 2]
      - symbol: 23
        coord: [4, 4]
      - symbol: 27
        coord: [4, 6]
      - symbol: 31
        coord: [4, 8]
      - symbol: 35
        coord: [6, 2]
      - symbol: 39
        coord: [6, 4]
      - symbol: 43
        coord: [6, 6]
      - symbol: 47
        coord: [6, 8]
      - symbol: 51
        coord: [8, 2]
      - symbol: 55
        coord: [8, 4]
      - symbol: 59
        coord: [8, 6]
      - symbol: 63
        coord: [8, 8]

### パイプライン

In [15]:
Code(filename=dist_pipeline_path, language="YAML")

source: OpenQASM2
input: data/tutorial_5.qasm
function: main
target: SC_LS_FIXED_V0
output: tutorial_5_dist.json
sc_ls_fixed_v0_topology: data/tutorial_5_dist_topology.yaml
sc_ls_fixed_v0_machine_type: DistributedDim2
sc_ls_fixed_v0_magic_generation_period: 15
sc_ls_fixed_v0_maximum_magic_state_stock: 10000
sc_ls_fixed_v0_entanglement_generation_period: 100
sc_ls_fixed_v0_maximum_entangled_state_stock: 10
sc_ls_fixed_v0_reaction_time: 1
sc_ls_fixed_v0_drop_rate: 0.1
sc_ls_fixed_v0_code_cycle_time_sec: 0.00000001
sc_ls_fixed_v0_physical_error_rate: 0.01
sc_ls_fixed_v0_allowed_failure_prob: 0.01
sc_ls_fixed_v0_pass:
  - sc_ls_fixed_v0::init_compile_info
  - sc_ls_fixed_v0::mapping
  - sc_ls_fixed_v0::routing

### 実行

In [16]:
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret asm -i tutorial_5_dist.json -o tutorial_5_dist.asm --print-metadata 1

2026-03-03 11:09:07 - INFO  - Load OpenQASM2.
2026-03-03 11:09:07 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:07 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:07 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:07 - INFO  - Run passes.
2026-03-03 11:09:07 - INFO  - Run InitCompileInfo
2026-03-03 11:09:07 - INFO  - Initialize compile information
2026-03-03 11:09:07 - INFO  - Run Mapping
2026-03-03 11:09:07 - INFO  - Run Routing
2026-03-03 11:09:07 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-03 11:09:07 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_dist.json


## PBC モードへのコンパイル
PBC は `sc_ls_fixed_v0_enable_pbc_mode: true` で有効化します。
このチュートリアルでは、制約を満たすように末尾で全量子ビットを測定した入力を使います。


In [17]:
pbc_pipeline_path = "data/tutorial_5_pbc_pipeline.yaml"

### 入力回路（PBC 用）

In [18]:
Code(filename="data/tutorial_5_pbc.qasm", language="ASM")

OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];
creg c[4];
h q[0];
h q[1];
h q[2];
h q[3];
t q[0];
cx q[0], q[1];
t q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
measure q[3] -> c[3];

### パイプライン

In [19]:
Code(filename=pbc_pipeline_path, language="YAML")

source: OpenQASM2
input: data/tutorial_5_pbc.qasm
function: main
target: SC_LS_FIXED_V0
output: tutorial_5_pbc.json
sc_ls_fixed_v0_topology: data/tutorial_5_dim2_topology.yaml
sc_ls_fixed_v0_machine_type: Dim2
sc_ls_fixed_v0_enable_pbc_mode: true
sc_ls_fixed_v0_magic_generation_period: 15
sc_ls_fixed_v0_maximum_magic_state_stock: 10000
sc_ls_fixed_v0_entanglement_generation_period: 100
sc_ls_fixed_v0_maximum_entangled_state_stock: 10
sc_ls_fixed_v0_reaction_time: 1
sc_ls_fixed_v0_drop_rate: 0.1
sc_ls_fixed_v0_code_cycle_time_sec: 0.00000001
sc_ls_fixed_v0_physical_error_rate: 0.01
sc_ls_fixed_v0_allowed_failure_prob: 0.01
sc_ls_fixed_v0_pass:
  - sc_ls_fixed_v0::init_compile_info
  - sc_ls_fixed_v0::mapping
  - sc_ls_fixed_v0::routing

### 実行

In [20]:
!qret compile --verbose --pipeline {pbc_pipeline_path}
!qret asm -i tutorial_5_pbc.json -o tutorial_5_pbc.asm --print-metadata 1

Code(filename="tutorial_5_pbc.asm", language="ASM")

2026-03-03 11:09:07 - INFO  - Load OpenQASM2.
2026-03-03 11:09:07 - INFO  - Build IR from OpenQASM2.
2026-03-03 11:09:07 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-03 11:09:07 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-03 11:09:07 - INFO  - Run passes.
2026-03-03 11:09:07 - INFO  - Run InitCompileInfo
2026-03-03 11:09:07 - INFO  - Initialize compile information
2026-03-03 11:09:07 - INFO  - Run Mapping
2026-03-03 11:09:07 - INFO  - Run Routing
2026-03-03 11:09:07 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-03 11:09:07 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_pbc.json


# ------------------------------------------------------------
# Compilation to SC_LS_FIXED_V0Machine by QRET.
# Notes:
# Assembly of SC_LS_FIXED_V0Machine has 10 reserved csymbols.
# - C0 is always 0.
# - C1 is always 1.
# - C2-C9 are reserved for future use.
# ------------------------------------------------------------
# header_schema: 0.1
# qret:
#   version: 1.0.0
# created_at: 2026-03-03T11:09:08
# csymbols:
#   constants: { C0: 0, C1: 1 }
#   reserved: [C2, C3, C4, C5, C6, C7, C8, C9]
# target:
#   type: Dim2
#   use_magic_state_cultivation: false
#   magic_factory_seed_offset: 0
#   magic_generation_period: 15  # cycles
#   prob_magic_state_creation: 1.00
#   maximum_magic_state_stock: 10000
#   entanglement_generation_period: 100  # cycles
#   maximum_entangled_state_stock: 10
#   reaction_time: 1  # cycles
# topology:
#   legend:
#     free: .
#     banned: x
#     qubit: Q
#     magic_factory: M
#     entanglement_factory: E
#   orientation:
#     origin: { x: 0, y: 0 }
#     x_increases: right
#     y_increases: up
#     row_order_in_layout: high_to_low
#   grid_0_1:
#     max_x: 10
#     max_y: 10
#     min_z: 0
#     max_z: 1
#     plane_0:
#       z: 0
#       free: 96
#       banned: 0
#       qubit: 0
#       magic_factory: 4
#       entanglement_factory: 0
#       layout:
#         y9: ..........
#         y8: ..........
#         y7: ..........
#         y6: ..........
#         y5: ..........
#         y4: ..........
#         y3: M.........
#         y2: M.........
#         y1: M.........
#         y0: M.........


ALLOCATE_MAGIC_FACTORY m0 (0,0,0)  # [beat: 0, z: 0];
ALLOCATE_MAGIC_FACTORY m1 (0,1,0)  # [beat: 0, z: 0];
ALLOCATE_MAGIC_FACTORY m2 (0,2,0)  # [beat: 0, z: 0];
ALLOCATE_MAGIC_FACTORY m3 (0,3,0)  # [beat: 0, z: 0];
ALLOCATE q0 (0,5,0) 0  # [beat: 0, z: 0];
ALLOCATE q1 (0,7,0) 0  # [beat: 16, z: 0];
ALLOCATE q2 (0,9,0) 0  # [beat: 22, z: 0];
ALLOCATE q3 (2,8,0) 0  # [beat: 22, z: 0];
ALLOCATE q4 (6,4,0) 0  # [beat: 0, z: 0];
ALLOCATE q5 (2,0,0) 0  # [beat: 0, z: 0];
ALLOCATE q6 (6,6,0) 0  # [beat: 0, z: 0];
ALLOCATE q7 (2,2,0) 0  # [beat: 0, z: 0];
ALLOCATE q8 (6,0,0) 0  # [beat: 22, z: 0];
ALLOCATE q9 (2,4,0) 0  # [beat: 22, z: 0];
ALLOCATE q10 (6,2,0) 0  # [beat: 22, z: 0];
ALLOCATE q11 (2,6,0) 0  # [beat: 22, z: 0];
MOVE_MAGIC m3 q4 [(1,3,0) (2,3,0) (3,3,0) (4,3,0) (5,3,0) (6,3,0)]  # [beat: 16, z: 0];
LATTICE_SURGERY [q0 q4] [X Z] [(0,4,0) (1,4,0) (2,4,0) (3,4,0) (4,4,0) (5,4,0)] c10  # [beat: 17, z: 0];
TWIST q5 0  # [beat: 0, z: 0];
LATTICE_SURGERY [q4 q5] [Z X] [(5,4,0) (5,3,0) (5,2,0) (5,1,0) (4,1,0) (3,1,0) (2,1,0)] c11  # [beat: 18, z: 0];
TWIST q5 0  # [beat: 19, z: 0];
MEAS_ZX q4 1 c12  # [beat: 19, z: 0];
MEAS_ZX q4 0 c13 [c10]  # [beat: 19, z: 0];
XOR [c1 c10] c15  # [beat: 18, z: 0];
MEAS_ZX q4 1 c14 [c15]  # [beat: 19, z: 0];
MOVE_MAGIC m1 q6 [(1,1,0) (2,1,0) (3,1,0) (3,2,0) (4,2,0) (5,2,0) (6,2,0) (7,2,0) (7,3,0) (7,4,0) (7,5,0) (6,5,0)]  # [beat: 16, z: 0];
LATTICE_SURGERY [q0 q1 q6] [X X Z] [(2,6,0) (1,6,0) (3,6,0) (4,6,0) (5,6,0) (0,6,0)] c16  # [beat: 18, z: 0];
TWIST q7 0  # [beat: 0, z: 0];
LATTICE_SURGERY [q6 q7] [Z X] [(5,6,0) (5,5,0) (5,4,0) (5,3,0) (4,3,0) (3,3,0) (2,3,0)] c17  # [beat: 19, z: 0];
TWIST q7 0  # [beat: 20, z: 0];
MEAS_ZX q6 1 c18  # [beat: 20, z: 0];
MEAS_ZX q6 0 c19 [c16]  # [beat: 20, z: 0];
XOR [c1 c16] c21  # [beat: 19, z: 0];
MEAS_ZX q6 1 c20 [c21]  # [beat: 20, z: 0];
MEAS_ZX q0 1 c22  # [beat: 19, z: 0];
LATTICE_SURGERY [q0 q1] [X X] [(0,6,0)] c23  # [beat: 19, z: 0];
MEAS_ZX q2 1 c24  # [beat: 22, z: 0];
MEAS_ZX q3 1 c25  # [beat: 22, z: 0];
DEALLOCATE q0  # [beat: 20, z: 0];
DEALLOCATE q1  # [beat: 20, z: 0];
DEALLOCATE q2  # [beat: 22, z: 0];
DEALLOCATE q3  # [beat: 22, z: 0];
DEALLOCATE q4  # [beat: 19, z: 0];
DEALLOCATE q5  # [beat: 21, z: 0];
DEALLOCATE q6  # [beat: 20, z: 0];
DEALLOCATE q7  # [beat: 22, z: 0];
DEALLOCATE q8  # [beat: 22, z: 0];
DEALLOCATE q9  # [beat: 22, z: 0];
DEALLOCATE q10  # [beat: 22, z: 0];
DEALLOCATE q11  # [beat:

## 次章への接続
次章では、この章で生成した `tutorial_5_*.json` を `profile` で解析し、
実行リソース（runtime・code distance・physical qubits など）を比較します。
